# 02 — Silver: Lead's slice (EC + DC + T&S + WS + T6 reconstruction + T9 bridge)

**Owner:** Lead • **Tier:** Heavy • **Phase:** 4 (Thu 14 May lunch – Sun 17 May)

## Scope
Cross-cutting Silver tables that don't fit into a single member's domain, plus the two complex Silver transforms (T6, T9):

- **EC (5):** silver_ec_orders, silver_ec_order_lines, silver_ec_order_status_history, silver_ec_order_status_codes, silver_ec_delivery_methods
- **DC (4):** silver_dc_shipments, silver_dc_delivery_events, silver_dc_event_types, silver_dc_carriers
- **T&S (4):** silver_ts_safety_reports, silver_ts_risk_signals, silver_ts_signal_types, silver_ts_report_reasons
- **WS (3):** silver_ws_sessions, silver_ws_page_events, silver_ws_event_types
- **Derived: silver_store_inventory_snapshots_reconstructed** (T6 output — sibling to M4's clean Silver)
- **silver_campaign_attribution_bridge** (T9 output)

## Anomalies in scope (9)
- **A5** (WH ATP>0 with physical=0) — detected during T6 setup
- **A7** (returned stock counted sellable before inspection)
- **A8** (missing snapshot days) — drives T6 reconstruction
- **A12** (NL relisting via image_hash + 30d window) — windowed query over nl_listings (M6's Silver)
- **A14** (delivery clock drift) — dc_shipments vs dc_delivery_events timestamp comparison
- **B1** (campaign attribution edge cases — 126 promo-less in-window orders)
- **B2** (partial refund period attribution defence)
- **B3** (175 PCK movements with NULL ref — defend interpretation)
- **B7** (BOPIS pickup gap analysis)

## Setup — Install connector + widgets (Free Edition serverless)

In [ ]:
%pip install -q snowflake-connector-python pandas
# dbutils.library.restartPython()  # SKIP on Free Edition — kernel restart hangs

In [ ]:
dbutils.widgets.text('sf_account',   'rhxendw-yb24678')
dbutils.widgets.text('sf_user',      'NEXAMART_LEAD')
dbutils.widgets.text('sf_password',  '')
dbutils.widgets.text('sf_warehouse', 'NEXAMART_WH')
dbutils.widgets.text('sf_role',      'NEXAMART_ENGINEER')

In [ ]:
from pyspark.sql import functions as F, Window
import sys
sys.path.append('/Workspace/Repos/<your-org>/nexamart-m1/notebooks/_shared')
from utils_dates     import parse_date, is_parse_failure
from utils_keys      import surrogate_key
from utils_anomaly   import add_anomaly_columns, flag, flag_date_parse_failure
from utils_snowflake import read_from_snowflake, write_to_snowflake

def read_bronze(t): return read_from_snowflake(spark, t, schema='NEXAMART_BRONZE')
def read_silver(t): return read_from_snowflake(spark, t, schema='NEXAMART_SILVER')
def write_silver(df, t):
    n = write_to_snowflake(df, t, schema='NEXAMART_SILVER')
    print(f'Wrote silver.{t} ({n} rows)')

## EC Silver — silver_ec_orders, silver_ec_order_lines, silver_ec_order_status_history

**TODO:** parse dates (iso_date for orders/lines, iso_timestamp for status_history); SK on order_id and (order_id, line_no); join customer SK from M2's silver_customer_master.

In [ ]:
# EC Silver — orders, lines, status_history + 2 lookups
# Dates: ec_orders.order_date = ISO (yyyy-MM-dd); created_at / status_timestamp = ISO timestamp.
# NOTE: customer_id 9999 collision (A11) is M2's; we preserve raw customer_id here.

# -- static lookups --
write_silver(add_anomaly_columns(read_bronze('ec_order_status_codes')), 'silver_ec_order_status_codes')
write_silver(add_anomaly_columns(read_bronze('ec_delivery_methods')),   'silver_ec_delivery_methods')

# -- silver_ec_orders --
o = (read_bronze('ec_orders')
     .withColumn('order_date_parsed', parse_date(F.col('order_date'), 'iso_date'))
     .withColumn('created_at_parsed', parse_date(F.col('created_at'), 'iso_timestamp'))
     .withColumn('order_sk', surrogate_key(F.col('order_id'))))
o = add_anomaly_columns(o)
o = flag(o, is_parse_failure(F.col('order_date_parsed'), F.col('order_date')),
         'DATE_PARSE_FAIL', 'FLAGGED_ANOMALY', 'UNRELIABLE')
write_silver(o, 'silver_ec_orders')

# -- silver_ec_order_lines --
ol = (read_bronze('ec_order_lines')
      .withColumn('order_line_sk', surrogate_key(F.col('order_id'), F.col('line_id'))))
write_silver(add_anomaly_columns(ol), 'silver_ec_order_lines')

# -- silver_ec_order_status_history --
sh = (read_bronze('ec_order_status_history')
      .withColumn('status_ts_parsed', parse_date(F.col('status_timestamp'), 'iso_timestamp'))
      .withColumn('status_history_sk', surrogate_key(F.col('history_id'))))
sh = add_anomaly_columns(sh)
sh = flag(sh, is_parse_failure(F.col('status_ts_parsed'), F.col('status_timestamp')),
          'DATE_PARSE_FAIL', 'FLAGGED_ANOMALY', 'UNRELIABLE')
write_silver(sh, 'silver_ec_order_status_history')

print('EC silvers written. orders=%d lines=%d status_hist=%d'
      % (o.count(), ol.count(), sh.count()))

## DC Silver + A14 detection (delivery clock drift)

**TODO:**
- Parse dates on dc_shipments (iso_date) and dc_delivery_events (iso_timestamp)
- Join shipments to events; flag where MIN(event_datetime) < created_datetime as DELIVERY_BEFORE_SHIP
- Within events for one shipment, flag DELIVERED events earlier than SHIPPED/PICKED_UP

In [ ]:
# DC Silver + A14 (delivery clock drift)
# Two detection contracts: strict DELIVERY_BEFORE_SHIP (DELIVERED < PICKED_UP, expect 18 shipments)
# and broad COURIER_CLOCK_DRIFT (any event before dc_shipments.created_datetime, expect 68 shipments).
# Correction: median transit is computed FROM the data (clean shipments), not assumed.

# -- lookups --
write_silver(add_anomaly_columns(read_bronze('dc_carriers')),    'silver_dc_carriers')
write_silver(add_anomaly_columns(read_bronze('dc_event_types')), 'silver_dc_event_types')

# -- silver_dc_shipments --
ship = (read_bronze('dc_shipments')
        .withColumn('created_dt_parsed', parse_date(F.col('created_datetime'), 'iso_timestamp'))
        .withColumn('shipment_sk', surrogate_key(F.col('shipment_id'))))
write_silver(add_anomaly_columns(ship), 'silver_dc_shipments')

# -- silver_dc_delivery_events + A14 --
ev = (read_bronze('dc_delivery_events')
      .withColumn('event_ts_parsed', parse_date(F.col('event_timestamp'), 'iso_timestamp'))
      .withColumn('event_sk', surrogate_key(F.col('event_id'))))

w = Window.partitionBy('shipment_id')
ev = (ev
      .withColumn('pickup_ts',    F.max(F.when(F.col('event_type_code') == 'PICKED_UP', F.col('event_ts_parsed'))).over(w))
      .withColumn('delivered_ts', F.min(F.when(F.col('event_type_code') == 'DELIVERED', F.col('event_ts_parsed'))).over(w)))

# data-driven median transit (hours) from CLEAN shipments (delivered >= pickup)
TRANSIT = (ev.filter((F.col('pickup_ts').isNotNull()) & (F.col('delivered_ts').isNotNull())
                     & (F.col('delivered_ts') >= F.col('pickup_ts')))
             .select('shipment_id',
                     ((F.col('delivered_ts').cast('long') - F.col('pickup_ts').cast('long')) / 3600.0).alias('h'))
             .distinct())
MEDIAN_TRANSIT_H = TRANSIT.approxQuantile('h', [0.5], 0.01)[0]
MANUAL_REVIEW_WINDOW_H = 72.0  # |delta| beyond this is too large to auto-correct
print('Data-driven median transit = %.1f h' % MEDIAN_TRANSIT_H)

ev = ev.join(ship.select('shipment_id', F.col('created_dt_parsed').alias('ship_created_ts')),
             on='shipment_id', how='left')
ev = add_anomaly_columns(ev)

# Broad (68 shipments): any event timestamp predates shipment creation
ev = flag(ev, F.col('event_ts_parsed') < F.col('ship_created_ts'),
          'COURIER_CLOCK_DRIFT', 'FLAGGED_AMBIGUOUS', 'INFERRED')

# Strict (18 shipments): a DELIVERED event earlier than the shipment's PICKED_UP
is_strict = (F.col('event_type_code') == 'DELIVERED') & (F.col('delivered_ts') < F.col('pickup_ts'))
ev = flag(ev, is_strict, 'DELIVERY_BEFORE_SHIP', 'FLAGGED_ANOMALY', 'UNRELIABLE')

# A14 correction: corrected = pickup + median transit when |delta| <= window; else manual review.
delta_h = (F.col('delivered_ts').cast('long') - F.col('pickup_ts').cast('long')) / 3600.0
ev = (ev
      .withColumn('corrected_event_datetime',
                  F.when(is_strict & (F.abs(delta_h) <= MANUAL_REVIEW_WINDOW_H),
                         (F.col('pickup_ts').cast('long') + F.lit(int(MEDIAN_TRANSIT_H * 3600))).cast('timestamp')))
      .withColumn('requires_manual_review',
                  is_strict & (F.abs(delta_h) > MANUAL_REVIEW_WINDOW_H)))
write_silver(ev, 'silver_dc_delivery_events')

strict_n = ev.filter(is_strict).select('shipment_id').distinct().count()
broad_n  = ev.filter(F.col('anomaly_reason_code').contains('COURIER_CLOCK_DRIFT')).select('shipment_id').distinct().count()
print('A14 strict DELIVERY_BEFORE_SHIP shipments = %d (expect 18)' % strict_n)
print('A14 broad  COURIER_CLOCK_DRIFT shipments = %d (expect 68)' % broad_n)

## T&S Silver + A7 + A12 detection

**TODO:**
- silver_ts_safety_reports, silver_ts_risk_signals (parse dates, SK)
- A7: detect returned stock counted sellable before inspection — cross-reference rr_return_receipts (M5's Silver) with si_inventory_movements
- A12: window over (seller_id, image_hash) on nl_listings (M6's Silver); previous SOLD/EXPIRED within 30d → RELISTED_AFTER_SOLD

In [ ]:
# T&S Silver (lead subset) + A7 + A12
# A7 and A12 are Lead-owned detections that run directly on the Bronze source tables
# (rr_return_receipts, nl_listings) — they do not depend on M5/M6 Silver.

# -- lookups / T&S facts in lead's scope --
for t in ['ts_safety_reports', 'ts_risk_signals', 'ts_signal_types', 'ts_report_reasons']:
    df = read_bronze(t)
    if t == 'ts_safety_reports':
        df = df.withColumn('reported_at_parsed', parse_date(F.col('reported_at'), 'iso_date'))   # YYYY-MM-DD
        df = df.withColumn('report_sk', surrogate_key(F.col('report_id')))
    if t == 'ts_risk_signals':
        df = df.withColumn('detected_at_parsed', parse_date(F.col('detected_at'), 'iso_date'))    # YYYY-MM-DD
        df = df.withColumn('signal_sk', surrogate_key(F.col('signal_id')))
    write_silver(add_anomaly_columns(df), f'silver_{t}')

# -- A7: restocked before inspection completed (expect 10) --
rr = (read_bronze('rr_return_receipts')
      .withColumn('received_date_parsed', parse_date(F.col('received_date'), 'iso_date'))
      .withColumn('receipt_sk', surrogate_key(F.col('receipt_id'))))
rr = add_anomaly_columns(rr)
rr = flag(rr, (F.col('inspection_status') == 'PENDING') & (F.col('restocked_qty') > 0),
          'RESTOCK_BEFORE_INSPECTION', 'FLAGGED_ANOMALY', 'UNRELIABLE')
write_silver(rr, 'silver_rr_return_receipts')
print('A7 RESTOCK_BEFORE_INSPECTION = %d (expect 10)'
      % rr.filter(F.col('anomaly_reason_code').contains('RESTOCK_BEFORE_INSPECTION')).count())

# -- A12: relisted after sold/expired (expect 1 metadata + 3 image-hash strict) --
nl = (read_bronze('nl_listings')
      .withColumn('created_at_parsed', parse_date(F.col('created_at'), 'iso_timestamp'))
      .withColumn('updated_at_parsed', parse_date(F.col('updated_at'), 'iso_timestamp'))
      .withColumn('listing_sk', surrogate_key(F.col('listing_id'))))

# strict self-join: ACTIVE relist whose (seller, image_hash) had a prior SOLD/EXPIRED
sold = (nl.filter(F.col('status_code').isin('SOLD', 'EXPIRED') & F.col('image_hash').isNotNull())
          .select(F.col('listing_id').alias('s_id'), F.col('seller_account_id').alias('s_sell'),
                  F.col('image_hash').alias('s_hash'), F.col('updated_at_parsed').alias('sold_ts')))
relist = (nl.filter((F.col('status_code') == 'ACTIVE') & F.col('image_hash').isNotNull())
            .select(F.col('listing_id').alias('r_id'), F.col('seller_account_id').alias('r_sell'),
                    F.col('image_hash').alias('r_hash'), F.col('created_at_parsed').alias('relist_ts')))
strict_ids = (relist.join(sold, (relist.r_sell == sold.s_sell) & (relist.r_hash == sold.s_hash)
                          & (relist.r_id != sold.s_id) & (relist.relist_ts > sold.sold_ts))
              .select('r_id').distinct())
strict_id_list = [r['r_id'] for r in strict_ids.collect()]

nl = add_anomaly_columns(nl)
nl = flag(nl, F.col('relist_count') > 0, 'RELISTED_AFTER_SOLD', 'FLAGGED_ANOMALY', 'INFERRED')      # metadata = 1
nl = flag(nl, F.col('listing_id').isin(strict_id_list), 'RELISTED_AFTER_SOLD', 'FLAGGED_ANOMALY', 'INFERRED')  # strict = 3
write_silver(nl, 'silver_nl_listings')
print('A12 metadata relist_count>0 = %d (expect 1); strict image-hash = %d (expect 3)'
      % (nl.filter(F.col('relist_count') > 0).count(), len(strict_id_list)))

## WS Silver — clickstream

**TODO:** silver_ws_sessions, silver_ws_page_events, silver_ws_event_types. Parse iso_timestamp. SK on session_id and (session_id, event_id).

In [ ]:
# WS Silver — clickstream (sessions, page_events, event_types)
# ws_sessions.session_start_ts / session_end_ts = ISO timestamp; BTS2024 filterable via utm_campaign.

write_silver(add_anomaly_columns(read_bronze('ws_event_types')), 'silver_ws_event_types')

ses = (read_bronze('ws_sessions')
       .withColumn('session_start_parsed', parse_date(F.col('session_start_ts'), 'iso_timestamp'))
       .withColumn('session_end_parsed',   parse_date(F.col('session_end_ts'),   'iso_timestamp'))
       .withColumn('session_sk', surrogate_key(F.col('session_id'))))
ses = add_anomaly_columns(ses)
ses = flag(ses, is_parse_failure(F.col('session_start_parsed'), F.col('session_start_ts')),
           'DATE_PARSE_FAIL', 'FLAGGED_ANOMALY', 'UNRELIABLE')
write_silver(ses, 'silver_ws_sessions')

pe = (read_bronze('ws_page_events')
      .withColumn('event_ts_parsed', parse_date(F.col('event_timestamp'), 'iso_timestamp'))
      .withColumn('page_event_sk', surrogate_key(F.col('session_id'), F.col('event_id'))))
pe = add_anomaly_columns(pe)
pe = flag(pe, is_parse_failure(F.col('event_ts_parsed'), F.col('event_timestamp')),
          'DATE_PARSE_FAIL', 'FLAGGED_ANOMALY', 'UNRELIABLE')
write_silver(pe, 'silver_ws_page_events')

print('WS silvers written. sessions=%d page_events=%d  (BTS2024 sessions=%d)'
      % (ses.count(), pe.count(), ses.filter(F.col('utm_campaign') == 'BTS2024').count()))

## T6 — ATP reconstruction → silver_store_inventory_snapshots_reconstructed

Read M4's clean silver_store_inventory_snapshots. For each (store, sku, calendar_day) with a gap (Stores 3, 7, 12 expected), reconstruct ATP via:
```
ATP_reconstructed = last_known_ATP
                  + Σreceives  (from gap_start to gap_day)
                  − Σissues
                  − Σdamage
                  + Σreturn_receives
```

Reconstructed rows: `data_quality_status='RECONSTRUCTED'`, `metric_certainty='INFERRED'`, `anomaly_reason_code='RECONSTRUCTED_SNAPSHOT'`. Write to a SIBLING table (not overwriting M4's clean Silver). Your fact_store_inventory_snapshot UNIONs both.

Also flag A8 (missing snapshot days) on the gap-day rows.

In [ ]:
# T6 — ATP reconstruction → silver_store_inventory_snapshots_reconstructed (sibling table)
# A8 evidence: stores 3, 7, 12 each miss Aug 1-7 (7 days) -> 21 store-days.
# DATA REALITY: the gap is co-missing in BOTH si_inventory_snapshots AND si_inventory_movements
# (those stores have no movement rows for Aug 1-7 either), so a movement-sum reconstruction is not
# possible. We instead LINEARLY INTERPOLATE between the two bracketing snapshots — 31/07/2024 and
# 08/08/2024 (an 8-day span) — and fall back to last-observation-carried-forward where a product has
# no 08/08 row. Rows are flagged RECONSTRUCTED / INFERRED.

GAP_STORES = [3, 7, 12]
# (snapshot_date, day-offset from 31/07): Aug1=1 .. Aug7=7; full bracket span = 8 days.
GAP_CAL = [('01/08/2024', 1), ('02/08/2024', 2), ('03/08/2024', 3), ('04/08/2024', 4),
           ('05/08/2024', 5), ('06/08/2024', 6), ('07/08/2024', 7)]
SPAN = 8.0

snap = read_bronze('si_inventory_snapshots')
b31 = (snap.filter(F.col('store_id').isin(GAP_STORES) & (F.col('snapshot_date') == '31/07/2024'))
           .select('store_id', 'product_code',
                   F.col('physical_qty').alias('p31'), F.col('sellable_qty').alias('s31')))
b08 = (snap.filter(F.col('store_id').isin(GAP_STORES) & (F.col('snapshot_date') == '08/08/2024'))
           .select('store_id', 'product_code',
                   F.col('physical_qty').alias('p08'), F.col('sellable_qty').alias('s08')))
b = b31.join(b08, ['store_id', 'product_code'], 'left')   # keep 31/07 products; 08/08 may be absent

cal = spark.createDataFrame(GAP_CAL, ['snapshot_date', 'day_off'])
spine = b.crossJoin(cal)

frac = F.col('day_off') / F.lit(SPAN)
recon = (spine
         .withColumn('physical_qty',
                     F.round(F.col('p31') + F.coalesce(F.col('p08') - F.col('p31'), F.lit(0)) * frac).cast('int'))
         .withColumn('sellable_qty',
                     F.round(F.col('s31') + F.coalesce(F.col('s08') - F.col('s31'), F.lit(0)) * frac).cast('int'))
         .withColumn('recon_method',
                     F.when(F.col('p08').isNotNull(), F.lit('LINEAR_INTERP')).otherwise(F.lit('LOCF')))
         .withColumn('is_reconstructed', F.lit(True))
         .withColumn('store_snap_sk', surrogate_key(F.col('store_id'), F.col('product_code'), F.col('snapshot_date')))
         .select('store_snap_sk', 'store_id', 'product_code', 'snapshot_date',
                 'physical_qty', 'sellable_qty', 'recon_method', 'is_reconstructed'))

recon = add_anomaly_columns(recon, default_status='RECONSTRUCTED', default_certainty='INFERRED')
recon = flag(recon, F.lit(True), 'RECONSTRUCTED_SNAPSHOT', 'RECONSTRUCTED', 'INFERRED')
recon = flag(recon, F.lit(True), 'MISSING_SNAPSHOT_DAY',  'RECONSTRUCTED', 'INFERRED')
write_silver(recon, 'silver_store_inventory_snapshots_reconstructed')

print('T6 reconstructed rows=%d across %d store-days (expect 21 = 3 stores x 7 days)'
      % (recon.count(), recon.select('store_id', 'snapshot_date').distinct().count()))

## T9 — silver_campaign_attribution_bridge

Join `silver_ws_sessions` (filtered to UTM=BTS2024) to subsequent `silver_ec_orders` for the same customer (loyalty match deterministic; probabilistic match via M2's identity bridge ≥ 0.90), where session_end ≤ 2h before order_placed.

Schema: bridge_key (SK), session_id, order_id, customer_master_key, match_confidence, session_end_at, order_placed_at, lag_minutes, attribution_rule (PROMO_CODE / SESSION_BRIDGE / BOPIS_WINDOW).

B1 candidate set surfaces here — 126 in-window EC orders without a promo code; defend attribution per case.

In [ ]:
# T9 — silver_campaign_attribution_bridge + B1
# B1 candidate set = Aug 8-28 EC orders with no promo code (126). Attributed subset = those whose
# session carried the BTS2024 UTM tag, via the DETERMINISTIC ec_orders.session_id = ws_sessions.session_id
# join (102). No M2 identity-bridge dependency for this headline number.

ec = read_bronze('ec_orders')
ws = read_bronze('ws_sessions')

in_window = (F.col('order_date') >= '2024-08-08') & (F.col('order_date') <= '2024-08-28')
candidate = ec.filter(in_window & F.col('promo_code_applied').isNull())

bts = (ws.filter(F.col('utm_campaign') == 'BTS2024')
         .select('session_id', 'session_end_ts', 'registered_customer_id').distinct())

bridge = (candidate.join(bts, on='session_id', how='inner')
          .withColumn('bridge_key', surrogate_key(F.col('session_id'), F.col('order_id')))
          .withColumn('attribution_rule', F.lit('SESSION_BRIDGE'))
          .withColumn('attribution_confidence', F.lit(0.85))
          .withColumn('session_end_at',  parse_date(F.col('session_end_ts'), 'iso_timestamp'))
          .withColumn('order_placed_at', parse_date(F.col('order_date'), 'iso_date'))
          .select('bridge_key', 'session_id', 'order_id',
                  F.col('registered_customer_id').alias('customer_id'),
                  'attribution_rule', 'attribution_confidence', 'session_end_at', 'order_placed_at'))
bridge = add_anomaly_columns(bridge, default_status='CLEAN', default_certainty='INFERRED')
bridge = flag(bridge, F.lit(True), 'ATTRIBUTION_SESSION_BRIDGE', 'CLEAN', 'INFERRED')
write_silver(bridge, 'silver_campaign_attribution_bridge')

print('B1 candidate (promo-less in-window) = %d (expect 126); attributed (BTS session) = %d (expect 102)'
      % (candidate.count(), bridge.select('order_id').distinct().count()))

## A5 detection — WH ATP > 0 with physical = 0

Profile silver_warehouse_inventory_snapshots (M4's output) for rows where atp_qty > 0 AND physical_qty = 0 in the campaign window. Flag with ATP_POSITIVE_PHYSICAL_ZERO.

In [ ]:
# A5 (WH ATP>0 & physical=0, expect 5) + B3 (WH movements NULL ref, expect 175 all PCK)
# wh_* are M4's domain; lead builds these here to seed the A5/B3 evidence while M4 Silver is pending.

wh = (read_bronze('wh_inventory_snapshots')
      .withColumn('snapshot_date_parsed', parse_date(F.col('snapshot_date'), 'iso_date'))
      .withColumn('wh_snap_sk', surrogate_key(F.col('warehouse_id'), F.col('sku'), F.col('snapshot_date'))))
wh = add_anomaly_columns(wh)
wh = flag(wh, (F.col('atp_qty') > 0) & (F.col('physical_qty') == 0),
          'ATP_POSITIVE_PHYSICAL_ZERO', 'FLAGGED_ANOMALY', 'UNRELIABLE')
write_silver(wh, 'silver_wh_inventory_snapshots')
print('A5 ATP_POSITIVE_PHYSICAL_ZERO = %d (expect 5)'
      % wh.filter(F.col('anomaly_reason_code').contains('ATP_POSITIVE_PHYSICAL_ZERO')).count())

whm = (read_bronze('wh_inventory_movements')
       .withColumn('movement_dt_parsed', parse_date(F.col('movement_datetime'), 'iso_timestamp'))
       .withColumn('wh_move_sk', surrogate_key(F.col('movement_id'))))
whm = add_anomaly_columns(whm)
whm = flag(whm, F.col('reference_number').isNull(),
           'MOVEMENT_NULL_REF', 'FLAGGED_AMBIGUOUS', 'INFERRED')
write_silver(whm, 'silver_wh_inventory_movements')
null_ref = whm.filter(F.col('anomaly_reason_code').contains('MOVEMENT_NULL_REF'))
print('B3 MOVEMENT_NULL_REF = %d (expect 175); distinct movement_type among them = %s (expect [PCK])'
      % (null_ref.count(), [r['movement_type'] for r in null_ref.select('movement_type').distinct().collect()]))

## B7 — BOPIS pickup gap analysis

Compare counts: silver_dc_delivery_events with event_type='BOPIS_COLLECTED' vs silver_ec_orders with delivery_method_code='BOPIS' AND order_status='DELIVERED'. The gap = orders marked DELIVERED without a pickup confirmation. Flag those EC orders with BOPIS_NO_PICKUP_EVENT.

In [ ]:
# B7 — BOPIS pickup gap (expect 25): shipments READY_FOR_PICKUP without a BOPIS_COLLECTED scan.
# Decision: treat the gap as fulfilled (scan-miss probable); flag collection_unconfirmed; exclude from BOPIS SLA.

ev = read_bronze('dc_delivery_events')
ready     = ev.filter(F.col('event_type_code') == 'READY_FOR_PICKUP').select('shipment_id').distinct()
collected = ev.filter(F.col('event_type_code') == 'BOPIS_COLLECTED').select('shipment_id').distinct()
gap_ship  = ready.join(collected, on='shipment_id', how='left_anti')   # ready but never collected
print('B7 READY=%d COLLECTED=%d GAP shipments=%d (expect 144/119/25)'
      % (ready.count(), collected.count(), gap_ship.count()))

ship = read_bronze('dc_shipments').select('shipment_id', 'order_reference')
gap_order_nums = [r['order_reference'] for r in
                  gap_ship.join(ship, on='shipment_id', how='inner').select('order_reference').distinct().collect()]

ecb = add_anomaly_columns(read_bronze('ec_orders'))
ecb = ecb.withColumn('collection_unconfirmed',
                     F.col('order_number').isin(gap_order_nums)
                     & (F.col('delivery_method_code') == 'BOPIS')
                     & (F.col('order_status') == 'DELIVERED'))
ecb = flag(ecb, F.col('collection_unconfirmed'),
           'BOPIS_NO_PICKUP_EVENT', 'FLAGGED_AMBIGUOUS', 'INFERRED')
write_silver(ecb, 'silver_ec_orders_bopis')
print('B7 flagged BOPIS orders (collection_unconfirmed=TRUE) = %d'
      % ecb.filter('collection_unconfirmed').count())